# 05 · Video pipeline design

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/05-video-pipeline-design.ipynb)

*Part III · group · 15 min*

> 🇪🇸 **Diseño de un pipeline de vídeo** — Diseñar la forma del tensor en cada etapa de dos sistemas de vídeo reales.

Design the tensor shape at every stage of two real video systems.

## What you will be able to do

- Design the tensor shape at each of five pipeline stages, for two different systems.
- Apply one ragged-length strategy from section 02 and give the exact batched shape.
- Decide where a new axis goes, and say how that choice affects the rest of the pipeline.

## Setup

Run this first. It installs and imports everything this notebook needs, and nothing else.

> 🇪🇸 Ejecuta esto primero: instala e importa todo lo que este cuaderno necesita.

In [ ]:
import numpy as np   # only needed for the padding-mask solution below

## Another discussion block

> 🇪🇸 Otro bloque de discusión: 10 minutos de diseño, 5 de puesta en común.
> No hay una única respuesta correcta.

Back to your breakout channel. 10 minutes design, 5 minutes share-back.
**There is no single correct answer.**

> Design the tensor shape at each stage — *raw file → decoded frames →
> preprocessed batch → model input → model output* — for **both** systems:
>
> - **Tech:** a short-video app computing one embedding per video from sampled
>   frames, to choose what to play next.
> - **Biotech:** a surgical-video model that labels the current phase of an
>   operation from an operating-room camera.

### The five questions

1. Sketch the shape at each of the five stages, for both. Where are they the
   same, and where must they differ?
2. Clips have different lengths — 30 seconds against 4 hours. Take one strategy
   your group proposed in section 02 and give the exact shape of the
   preprocessed batch. What does an invented or wasted value in that tensor
   represent?
3. The surgical system adds **three camera angles** recording at once. Where does
   that axis go, and why does its position change how easy the rest of the
   pipeline is to write?
4. The recommender samples 8 frames out of 900. Which operation from section 03
   does that, and what is lost?
5. Both systems must decide **which frames matter most**. What kind of mechanism
   could learn that weighting?

## Exercise 1 — sketch the two pipelines

> 🇪🇸 Dibuja las dos tuberías, etapa por etapa.

Use comments. The point is the shapes and what each axis counts, not running
code.

In [ ]:
# TODO 1: Fill in the shape at each stage for BOTH systems. Next to each,
#         write what the axes count.

# --- Tech: short-video recommender, one embedding per video -------------------
# raw file          : ...
# decoded frames    : ...
# preprocessed batch: ...
# model input       : ...
# model output      : ...

# --- Biotech: surgical phase labelling, one label per timestep ---------------
# raw file          : ...
# decoded frames    : ...
# preprocessed batch: ...
# model input       : ...
# model output      : ...

In [ ]:
#@title Solution — try it yourself first { display-mode: 'form' }
# One defensible answer. Your group's may differ and still be right — what
# matters is that you can say what every axis COUNTS.

# --- Tech: short-video recommender -------------------------------------------
# raw file          : bytes on disk, no shape yet
# decoded frames    : (900, 1080, 1920, 3)     T, H, W, C  — every frame
# preprocessed batch: (32, 8, 224, 224, 3)     N, T, H, W, C — 8 sampled frames
# model input       : (32, 8, 224, 224, 3)
# model output      : (32, 512)                N, embedding — TIME IS GONE,
#                                              collapsed into one vector per video

# --- Biotech: surgical phase labelling ---------------------------------------
# raw file          : bytes on disk
# decoded frames    : (432000, 1080, 1920, 3)  4 hours at 30fps
# preprocessed batch: (4, 64, 224, 224, 3)     N, T, H, W, C — a sliding window
# model input       : (4, 64, 224, 224, 3)
# model output      : (4, 64, 12)              N, T, classes — ONE LABEL PER
#                                              TIMESTEP, so time SURVIVES

# The five stages look alike until the output. The recommender destroys the time
# axis on purpose; the surgical model must keep it, because the answer to
# "what phase are we in?" changes during the operation.

## Exercise 2 — ragged lengths, and the extra camera

> 🇪🇸 Longitudes distintas y la cámara adicional.

In [ ]:
# TODO 2: Take ONE ragged-length strategy from section 02 (pad + mask, or
#         sample a fixed number of frames). Give the exact shape of the
#         preprocessed batch for 4 clips of 30s, 45s, 2min and 4h at 30 fps.
#         What does an invented or wasted value in that tensor represent?

# TODO 3: Add three camera angles to the surgical system. Write the batch shape
#         with the camera axis in two different positions, and say which makes
#         the rest of the pipeline easier to write.

In [ ]:
#@title Solution — try it yourself first { display-mode: 'form' }
# TODO 2 — padding to the longest clip is the honest disaster:
#   longest = 4h at 30fps = 432,000 frames
#   padded batch: (4, 432000, 224, 224, 3)  ~ 5.8e11 values. Not possible.
#   An invented value is a frame that was never recorded. The mask is what stops
#   the model from learning from footage that does not exist.
#
# Sampling a fixed 64 frames per clip:
#   batch: (4, 64, 224, 224, 3)  — fits easily.
#   Nothing is invented; a great deal is DISCARDED, and the 4-hour clip is
#   sampled 500x more sparsely than the 30-second one. That bias is real.
#
# The full padded tensor above is too big to build, but the MASK is not — one
# bit per frame instead of a (224, 224, 3) image per frame — so build that:

durations_s = [30, 45, 2 * 60, 4 * 60 * 60]     # 30s, 45s, 2min, 4h
lengths = [int(d * 30) for d in durations_s]    # frames at 30fps
T_max = max(lengths)

mask = np.zeros((4, T_max), dtype=bool)
for i, n in enumerate(lengths):
    mask[i, :n] = True                          # True = a real, recorded frame

wasted = 1 - mask.sum() / mask.size
print(lengths, "padded to", T_max, "-> wasted fraction", f"{wasted:.1%}")

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(10, 2.5))
ax.imshow(mask, aspect="auto", cmap="Greys", interpolation="nearest")
ax.set_yticks(range(4)); ax.set_yticklabels(["30s", "45s", "2min", "4h"])
ax.set_xlabel("frame index (0 to 432,000)")
ax.set_title(f"real frames (black) vs invented padding (white) — "
             f"{wasted:.1%} of this tensor is invented")
plt.tight_layout()
plt.show()

# TODO 3 — two placements:
#   (N, CAM, T, H, W, C)  = (4, 3, 64, 224, 224, 3)
#   (N * CAM, T, H, W, C) = (12, 64, 224, 224, 3)
#
# The second is easier: every existing per-video operation keeps working
# unchanged, because the camera axis has been folded into the batch axis — and
# a batch axis is exactly the axis whose order does not matter. You only need
# the first form when the model must COMBINE the angles, at which point you
# must unfold back and the shape bookkeeping becomes yours to get right.

## Share-back

> 🇪🇸 Puesta en común.

**Q4** — sampling 8 frames from 900 is *fancy indexing*, exactly section 03's
TODO 3: `frames[idx]` where `idx` is an array of positions. What is lost is
everything between the samples — and for a 4-hour surgical video, that is almost
all of it. That is why the surgical system uses a sliding window instead.

**Q5** — **attention**. It learns a weight per position from the data itself,
rather than you choosing which frames matter in advance. Take-home B in section
11 builds it from two `einsum` calls, and the padding mask from Q2 above turns
out to be the same mask attention needs.

### The thread running through both discussions

Section 02 asked what an axis *means*. This block asks where to *put* it. The
answer is the same in both: an axis whose order carries no information (batch,
camera) can be folded, shuffled and merged freely; an axis whose order **is** the
information (time) cannot.

---

## Done with this section

Next up: **06 · Contraction with einsum** — [open in Colab](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/06-contraction-with-einsum.ipynb).

[← Back to the workshop site](https://project-delphi.github.io/tensors-workshop/) · [All notebooks](https://project-delphi.github.io/tensors-workshop/notebooks.html) · [Handbook](https://project-delphi.github.io/tensors-workshop/tensors_workshop_plan_with_quizzes.html)